In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os, json, glob, re
import pandas as pd

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

JSON_LABEL_DIR = "../json_label"
JSON_EMB_DIR = "../json_emb"
MODEL_DIR = "../src/ml_model/decision_tree_models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Map tên môn trong json_label -> key tương ứng trong json_emb
SUBJECT_TO_EMB_KEY = {
    "tiếng việt": "vietnamese_comment",
    "toán": "mathematics_comment",
    "th-cn (tin học)": "informatics_comment",
    "tin học": "informatics_comment",
    "ngoại ngữ tiếng anh": "english_comment",
    "tiếng anh": "english_comment",
    "nghệ thuật (âm nhạc)": "music_comment",
    "âm nhạc": "music_comment",
    "nghệ thuật (mĩ thuật)": "arts_comment",
    "mĩ thuật": "arts_comment",
    "đạo đức": "civics_comment",
    "giáo dục thể chất": "physical_education_comment",
    "hoạt động trải nghiệm": "experiential_activities_comment",
    "tự nhiên và xã hội": "nature_and_society_comment",
    "khoa học": "science_comment",
    "lịch sử và địa lí": "history_geography_comment",
    "công nghệ": "technology_comment",
}

TARGET_LEVELS = {
    "Biết": 0,
    "Hiểu": 1,
    "Vận dụng": 2,
    "Không có nhận xét môn học": 3,
}

REVERSE_TARGET_LEVELS = {v: k for k, v in TARGET_LEVELS.items()}

In [3]:
glob.glob(os.path.join(JSON_LABEL_DIR, "*_labeled.json"))[0]

'../json_label\\công nghệ_labeled.json'

In [4]:
def normalize_name(path):
    name = os.path.basename(path)
    name = name.replace("_labeled.json", "")
    name = name.replace("_", " ")
    return name.strip().lower()
normalize_name('../json_label\\công nghệ_labeled.json')

'công nghệ'

In [5]:
def load_labels():
    """
    Return:
    {
        subject_name: {
            student_code: target_level_text
        }
    }
    """
    all_labels = {}

    for path in glob.glob(os.path.join(JSON_LABEL_DIR, "*_labeled.json")):
        subject = normalize_name(path)

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        subject_labels = {}
        for item in data:
            student_code = item.get("student_code")
            level = item.get("level")

            if student_code and level:
                subject_labels[student_code] = level

        all_labels[subject] = subject_labels

    return all_labels
load_labels()

{'công nghệ': {'20252026.03.Ba1.010': 'Hiểu',
  '20252026.03.Ba1.011': 'Vận dụng',
  '20252026.03.Ba1.012': 'Hiểu',
  '20252026.03.Ba1.013': 'Vận dụng',
  '20252026.03.Ba1.014': 'Hiểu',
  '20252026.03.Ba1.015': 'Vận dụng',
  '20252026.03.Ba1.016': 'Vận dụng',
  '20252026.03.Ba1.017': 'Vận dụng',
  '20252026.03.Ba1.018': 'Vận dụng',
  '20252026.03.Ba1.019': 'Vận dụng',
  '20252026.03.Ba1.001': 'Vận dụng',
  '20252026.03.Ba1.020': 'Vận dụng',
  '20252026.03.Ba1.021': 'Vận dụng',
  '20252026.03.Ba1.022': 'Vận dụng',
  '20252026.03.Ba1.023': 'Vận dụng',
  '20252026.03.Ba1.024': 'Hiểu',
  '20252026.03.Ba1.025': 'Vận dụng',
  '20252026.03.Ba1.026': 'Vận dụng',
  '20252026.03.Ba1.027': 'Vận dụng',
  '20252026.03.Ba1.028': 'Hiểu',
  '20252026.03.Ba1.002': 'Hiểu',
  '20252026.03.Ba1.004': 'Vận dụng',
  '20252026.03.Ba1.005': 'Vận dụng',
  '20252026.03.Ba1.006': 'Vận dụng',
  '20252026.03.Ba1.007': 'Hiểu',
  '20252026.03.Ba1.008': 'Vận dụng',
  '20252026.03.Ba1.009': 'Vận dụng',
  '20252026.03.B

In [6]:
def load_emb_features():
    """
    Return list of rows:
    student_code, emb_key, remembering_confident, remembering_level, ...
    """
    rows = []

    for path in glob.glob(os.path.join(JSON_EMB_DIR, "*.json")):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        student_code = data.get("student_code")
        if not student_code:
            continue

        for emb_key, value in data.items():
            if emb_key == "student_code":
                continue

            if not isinstance(value, dict):
                continue

            subject = value.get("subject")

            row = {
                "student_code": student_code,
                "emb_key": emb_key,
                "subject": subject,

                "remembering_confident": value.get("remembering", {}).get("confident"),
                "remembering_level": value.get("remembering", {}).get("level"),

                "understanding_confident": value.get("understanding", {}).get("confident"),
                "understanding_level": value.get("understanding", {}).get("level"),

                "applying_confident": value.get("applying", {}).get("confident"),
                "applying_level": value.get("applying", {}).get("level"),
            }

            rows.append(row)

    return pd.DataFrame(rows)
load_emb_features()

,student_code,emb_key,subject,remembering_confident,remembering_level,understanding_confident,understanding_level,applying_confident,applying_level
0,20252026.01.Mo1.001,vietnamese_comment,Tiếng Việt,0.754022,1,0.589424,1,0.810278,1
1,20252026.01.Mo1.001,mathematics_comment,Toán,0.623037,1,0.632618,1,0.663376,1
2,20252026.01.Mo1.001,informatics_comment,Tin học,0.449211,1,0.563636,1,0.529778,1
3,20252026.01.Mo1.001,english_comment,Tiếng Anh,0.744320,1,0.558076,1,0.610361,1
4,20252026.01.Mo1.001,music_comment,Âm nhạc,0.848767,1,0.683486,1,0.439514,1
...,...,...,...,...,...,...,...,...,...
6140,20252026.05.Na4.030,music_comment,Âm nhạc,0.703437,1,0.705508,1,0.752987,1
6141,20252026.05.Na4.030,arts_comment,Mĩ thuật,0.544041,1,0.575978,1,0.607174,1
6142,20252026.05.Na4.030,civics_comment,Đạo đức,0.379146,1,0.398043,1,0.494647,1
6143,20252026.05.Na4.030,physical_education_comment,Giáo dục thể chất,0.519547,1,0.369341,1,0.677869,1


In [7]:
def train_one_subject(subject_name, labels_by_student, emb_df):
    emb_key = SUBJECT_TO_EMB_KEY.get(subject_name)

    if emb_key is None:
        print(f"[SKIP] Chưa map môn: {subject_name}")
        return None

    df = emb_df[emb_df["emb_key"] == emb_key].copy()

    df["target_text"] = df["student_code"].map(labels_by_student)
    df = df.dropna(subset=["target_text"])

    df = df[df["target_text"].isin(TARGET_LEVELS)]

    feature_cols = [
        "remembering_confident",
        "remembering_level",
        "understanding_confident",
        "understanding_level",
        "applying_confident",
        "applying_level",
    ]

    df = df.dropna(subset=feature_cols)

    if len(df) < 5:
        print(f"[SKIP] {subject_name}: quá ít dữ liệu ({len(df)} dòng)")
        return None

    X = df[feature_cols]
    y = df["target_text"].map(TARGET_LEVELS)

    # Nếu số lượng class quá ít thì không stratify
    stratify = y if y.value_counts().min() >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=stratify,
    )

    clf = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=3,
        random_state=42,
        class_weight="balanced",
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    print("\n" + "=" * 80)
    print(f"Môn: {subject_name}")
    print(f"Số mẫu: {len(df)}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[REVERSE_TARGET_LEVELS[i] for i in sorted(set(y_test) | set(y_pred))],
            zero_division=0,
        )
    )

    print("Cây quyết định:")
    print(
        export_text(
            clf,
            feature_names=feature_cols,
        )
    )

    model_path = os.path.join(MODEL_DIR, f"{subject_name}.joblib")
    joblib.dump(
        {
            "model": clf,
            "feature_cols": feature_cols,
            "target_mapping": TARGET_LEVELS,
            "reverse_target_mapping": REVERSE_TARGET_LEVELS,
            "emb_key": emb_key,
        },
        model_path,
    )

    print(f"Đã lưu model: {model_path}")

    return clf

In [8]:
labels = load_labels()
emb_df = load_emb_features()

print("Số dòng embedding:", len(emb_df))
print("Các môn label:", list(labels.keys()))

for subject_name, labels_by_student in labels.items():
    train_one_subject(subject_name, labels_by_student, emb_df)

Số dòng embedding: 6145
Các môn label: ['công nghệ', 'giáo dục thể chất', 'hoạt động trải nghiệm', 'khoa học', 'lịch sử và địa lí', 'nghệ thuật (mĩ thuật)', 'nghệ thuật (âm nhạc)', 'ngoại ngữ tiếng anh', 'th-cn (tin học)', 'tiếng việt', 'toán', 'tự nhiên và xã hội', 'đạo đức']

Môn: công nghệ
Số mẫu: 363
Accuracy: 0.8356
              precision    recall  f1-score   support

        Biết       0.83      0.71      0.77         7
        Hiểu       0.75      0.56      0.64        16
    Vận dụng       0.85      0.94      0.90        50

    accuracy                           0.84        73
   macro avg       0.81      0.74      0.77        73
weighted avg       0.83      0.84      0.83        73

Cây quyết định:
|--- applying_confident <= 0.77
|   |--- applying_confident <= 0.50
|   |   |--- remembering_confident <= 0.37
|   |   |   |--- class: 2
|   |   |--- remembering_confident >  0.37
|   |   |   |--- applying_confident <= 0.49
|   |   |   |   |--- class: 0
|   |   |   |--- applying_